In [ ]:
!pip install timm einops -q

import os, json, zipfile
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision.transforms as transforms
from PIL import Image
from timm.models.vision_transformer import Block, PatchEmbed
from einops import rearrange
import matplotlib.pyplot as plt

def set_seed(seed=123):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = True

set_seed(123)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
class CelebAMAEDataset(Dataset):
    def __init__(self, data_dir, split='train', train_ratio=0.9):
        base_dir = Path(data_dir)
        possible_paths = [
            base_dir / "data" / "processed",
            base_dir / "processed",
            base_dir
        ]
        self.data_dir = None
        for path in possible_paths:
            if (path / "images").exists():
                self.data_dir = path
                break
        if self.data_dir is None:
            raise FileNotFoundError("Cannot find images folder")

        self.image_paths = sorted((self.data_dir / "images").glob("*.jpg"))
        split_idx = int(len(self.image_paths) * train_ratio)
        self.image_paths = self.image_paths[:split_idx] if split == 'train' else self.image_paths[split_idx:]

        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]) if split == 'train' else transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        self.has_region_masks = (self.data_dir / "region_masks").exists()
        print(f" {split}: {len(self.image_paths)} samples")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        region_mask = torch.zeros(196, dtype=torch.long)
        return img, region_mask


def get_dataloaders(data_dir, batch_size=64, num_workers=4):
    train_ds = CelebAMAEDataset(data_dir, split='train')
    val_ds   = CelebAMAEDataset(data_dir, split='val')
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True,
                              persistent_workers=True, prefetch_factor=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False,
                              num_workers=num_workers, pin_memory=True,
                              persistent_workers=True, prefetch_factor=2)
    print(f" Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")
    return train_loader, val_loader

DATA_DIR = "/kaggle/input/datasets/doandangkhoa/celeba-processed-224"  # sửa đúng path
train_loader, val_loader = get_dataloaders(DATA_DIR)

In [ ]:
class BlockwiseMasking:
    def __init__(self, mask_ratio=0.75, block_size=2):  
        self.mask_ratio  = mask_ratio
        self.num_patches = 196
        self.block_size  = block_size
        self.grid_size   = 14

    def generate_mask(self, batch_size, region_masks=None):
        num_blocks_per_side = self.grid_size // self.block_size   
        num_blocks_total    = num_blocks_per_side ** 2            
        num_to_mask         = int(self.mask_ratio * num_blocks_total)  

        masks = []
        for _ in range(batch_size):
            block_indices = torch.randperm(num_blocks_total)[:num_to_mask]
            mask = torch.zeros(self.grid_size, self.grid_size, dtype=torch.bool)
            for idx in block_indices:
                bi, bj = idx // num_blocks_per_side, idx % num_blocks_per_side
                si, sj = bi * self.block_size, bj * self.block_size
                mask[si:si+self.block_size, sj:sj+self.block_size] = True
            masks.append(mask.flatten())
        return torch.stack(masks)

In [ ]:
class MAEEncoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=384, depth=6, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size,
                                      in_chans=in_chans, embed_dim=embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.blocks      = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                  qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        w = self.patch_embed.proj.weight.data
        nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        self.apply(self._init_block_weights)

    def _init_block_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, mask):
        x            = self.patch_embed(x) + self.pos_embed
        num_visible  = (~mask)[0].sum().item()
        vis_idx      = (~mask).float().argsort(dim=1, descending=True)[:, :num_visible]
        x_visible    = torch.gather(x, 1, vis_idx.unsqueeze(-1).expand(-1, -1, x.shape[-1]))
        for block in self.blocks:
            x_visible = block(x_visible)
        return self.norm(x_visible), vis_idx

print(" Encoder ready!")

In [ ]:
class MAEDecoder(nn.Module):
    def __init__(self, num_patches=196, patch_size=16, in_chans=3,
                 encoder_embed_dim=384, decoder_embed_dim=256,
                 decoder_depth=4, decoder_num_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.num_patches       = num_patches
        self.decoder_embed     = nn.Linear(encoder_embed_dim, decoder_embed_dim)
        self.mask_token        = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches, decoder_embed_dim))
        self.decoder_blocks    = nn.ModuleList([
            Block(dim=decoder_embed_dim, num_heads=decoder_num_heads, mlp_ratio=mlp_ratio,
                  qkv_bias=True, norm_layer=nn.LayerNorm)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size ** 2 * in_chans)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        self.apply(self._init_block_weights)

    # FIX: dùng function riêng thay vì lambda
    def _init_block_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x_visible, vis_idx, mask):
        x_visible = self.decoder_embed(x_visible)
        D         = x_visible.shape[-1]
        B         = x_visible.shape[0]
        x_full    = self.mask_token.to(x_visible.dtype).expand(B, self.num_patches, -1).clone()
        x_full.scatter_(1, vis_idx.unsqueeze(-1).expand(-1, -1, D), x_visible)
        x_full    = x_full + self.decoder_pos_embed.to(x_visible.dtype)
        for block in self.decoder_blocks:
            x_full = block(x_full)
        return self.decoder_pred(self.decoder_norm(x_full))

print("✓ Decoder ready!")

In [ ]:
class MAE(nn.Module):
    def __init__(self, patch_size=16, encoder_embed_dim=384, encoder_depth=6,
                 encoder_num_heads=6, decoder_embed_dim=256, decoder_depth=4,
                 decoder_num_heads=8, mask_ratio=0.75, norm_pix_loss=True):
        super().__init__()
        self.patch_size    = patch_size
        self.num_patches   = (224 // patch_size) ** 2
        self.norm_pix_loss = norm_pix_loss
        self.masker        = BlockwiseMasking(mask_ratio)
        self.encoder       = MAEEncoder(embed_dim=encoder_embed_dim, depth=encoder_depth,
                                        num_heads=encoder_num_heads)
        self.decoder       = MAEDecoder(encoder_embed_dim=encoder_embed_dim,
                                        decoder_embed_dim=decoder_embed_dim,
                                        decoder_depth=decoder_depth,
                                        decoder_num_heads=decoder_num_heads)

    def patchify(self, imgs):
        p = self.patch_size
        return rearrange(imgs, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=p, p2=p)

    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(x.shape[1] ** 0.5)
        return rearrange(x, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)', h=h, w=w, p1=p, p2=p)

    def forward(self, imgs, region_masks=None):
        mask            = self.masker.generate_mask(imgs.shape[0]).to(imgs.device)
        latent, vis_idx = self.encoder(imgs, mask)
        pred            = self.decoder(latent, vis_idx, mask)
        target          = self.patchify(imgs)
        if self.norm_pix_loss:
            mean   = target.mean(dim=-1, keepdim=True)
            var    = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6).sqrt()
        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask.float()).sum() / (mask.float().sum() + 1e-6)
        return loss.unsqueeze(0), pred, mask

model = MAE().to(device)
# 1. IN THÔNG TIN VÀ TÍNH THAM SỐ 
print(f" Strategy : {type(model.masker).__name__}")
print(f"Mask_ratio     : {model.masker.mask_ratio}")      

enc_params   = sum(p.numel() for p in model.encoder.parameters()) / 1e6
dec_params   = sum(p.numel() for p in model.decoder.parameters()) / 1e6
total_params = enc_params + dec_params
print(f" MAE model ready!")
print(f"  Encoder: {enc_params:.1f}M | Decoder: {dec_params:.1f}M | Total: {total_params:.1f}M")

# 2. BỌC MULTI-GPU (DATAPARALLEL) 
if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

# 3. CHẠY SMOKE TEST
test_imgs = torch.randn(4, 3, 224, 224).to(device)
test_rm   = torch.randint(0, 2, (4, 196)).to(device)
with torch.no_grad():
    loss, pred, mask = model(test_imgs, test_rm)
    
loss = loss.mean() 

print(f"  Smoke test loss: {loss.item():.4f} | pred: {pred.shape}")
del test_imgs, test_rm, loss, pred, mask
torch.cuda.empty_cache()

In [ ]:
def cosine_scheduler(base_value, final_value, epochs, warmup_epochs=10):
    warmup  = np.linspace(0, base_value, warmup_epochs)
    iters   = np.arange(epochs - warmup_epochs)
    cosine  = final_value + 0.5*(base_value - final_value)*(1 + np.cos(np.pi*iters/len(iters)))
    return np.concatenate((warmup, cosine))

def safe_save(ckpt, path, backup=None):
    path = Path(path)
    tmp  = path.with_suffix('.tmp')
    torch.save(ckpt, tmp)
    tmp.replace(path)
    if backup:
        shutil.copy2(path, backup)

def train_one_epoch(model, loader, optimizer, scaler, device, epoch):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch:3d}")
    for imgs, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda"):
            loss, _, _ = model(imgs)
        
        loss = loss.mean() 
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total += loss.item() * imgs.shape[0]
        n     += imgs.shape[0]
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'avg': f"{total/n:.4f}"})
    return total / n

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total, n = 0.0, 0
    for imgs, _ in tqdm(loader, desc="[Val]", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        with autocast("cuda"):
            loss, _, _ = model(imgs)
            
        loss = loss.mean()
        total += loss.item() * imgs.shape[0]
        n     += imgs.shape[0]
    return total / n

In [ ]:
def train_mae_blockwise(model, train_loader, val_loader,
                     epochs=100, base_lr=1.5e-4, min_lr=1e-6,
                     warmup_epochs=10, weight_decay=0.05,
                     save_dir="/kaggle/working", model_name="mae_blockwise",
                     resume_from=None, save_every=5, device='cuda'):

    save_dir = Path(save_dir)
    ckpt_dir = save_dir / "checkpoints" / model_name
    tmp_dir  = Path(f"/tmp/{model_name}")
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, betas=(0.9, 0.95), weight_decay=weight_decay)
    scaler    = GradScaler("cuda")
    
    start_epoch = 0
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    # --- RESUME ---
    if resume_from and Path(resume_from).exists():
        print(f"Đang khôi phục từ: {resume_from}")
        ckpt = torch.load(resume_from, map_location='cpu', weights_only=False)
        
        # Clean module.
        state_dict = ckpt['model']
        clean_state_dict = {}
        for k, v in state_dict.items():
            clean_key = k.replace('module.', '') if k.startswith('module.') else k
            clean_state_dict[clean_key] = v
        
        # Xử lý trường hợp model bọc DataParallel
        if hasattr(model, 'module'):
            model.module.load_state_dict(clean_state_dict)
        else:
            model.load_state_dict(clean_state_dict)
            
        optimizer.load_state_dict(ckpt['optimizer'])
        scaler.load_state_dict(ckpt['scaler'])
        
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        history = ckpt.get('history', history)
        
        current_lr = optimizer.param_groups[0]['lr']
        remaining_epochs = epochs - start_epoch
        
        if remaining_epochs <= 0:
            print("Đã hoàn thành số epoch yêu cầu. Dừng train.")
            return model, history
            
        lr_schedule = cosine_scheduler(current_lr, min_lr, remaining_epochs, warmup_epochs=0)
        print(f"Tiếp tục từ Epoch {start_epoch} | Best Val Loss hiện tại: {best_val_loss:.4f} | LR: {current_lr:.2e}")
    else:
        lr_schedule = cosine_scheduler(base_lr, min_lr, epochs, warmup_epochs)
        print("Bắt đầu huấn luyện từ đầu (Epoch 0)")


    for epoch in range(start_epoch, epochs):

        is_resuming = resume_from and Path(resume_from).exists()
        
        lr = lr_schedule[epoch - start_epoch] if is_resuming else lr_schedule[epoch]
            
        for pg in optimizer.param_groups: pg['lr'] = lr

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, epoch)
        val_loss   = validate(model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(lr)

        is_best = val_loss < best_val_loss
        if is_best: best_val_loss = val_loss

        print(f"Epoch {epoch:3d} | train: {train_loss:.4f} | val: {val_loss:.4f} | lr: {lr:.2e}" +
              (" ← best" if is_best else ""))

        model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()

        # Lưu Checkpoints
        ckpt_data = {'epoch': epoch, 'model': model_state,
                     'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                     'val_loss': val_loss, 'best_val_loss': best_val_loss, 'history': history}

        safe_save(ckpt_data, ckpt_dir / "latest.pth", tmp_dir / "latest.pth")
        
        if is_best:
            safe_save(ckpt_data, ckpt_dir / "best.pth", tmp_dir / "best.pth")
        if (epoch + 1) % save_every == 0:
            safe_save(ckpt_data, ckpt_dir / f"epoch_{epoch:03d}.pth", tmp_dir / f"epoch_{epoch:03d}.pth")
            
        torch.cuda.empty_cache()

    print(f"\n✓ Training complete! Best val loss: {best_val_loss:.4f}")
    return model, history, ckpt_dir

In [ ]:
# ============================================
# 3. THỰC THI (CHẠY TRAINING & NÉN FILE)
# ============================================

RESUME_PATH = None
# RESUME_PATH = "/kaggle/working/checkpoints/mae_blockwise/latest.pth"

trained_model, history, ckpt_dir = train_mae_blockwise(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    resume_from=RESUME_PATH,
    device=device
)

# --- Lưu Weights sạch và Nén kết quả ---
SAVE_DIR = Path("/kaggle/working")
weights_path = SAVE_DIR / "mae_blockwise_weights.pth"
zip_path = SAVE_DIR / "mae_blockwise_result.zip"

if hasattr(trained_model, 'module'):
    torch.save(trained_model.module.state_dict(), weights_path)
else:
    torch.save(trained_model.state_dict(), weights_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    if (ckpt_dir / "best.pth").exists():
        zf.write(ckpt_dir / "best.pth", "best.pth")
    zf.write(weights_path, "mae_blockwise_weights.pth")

print(f"Đã lưu và nén xong! Hãy tải '{zip_path.name}' từ bảng Output bên phải.")